# 01 - Data Exploration

This notebook explores the raw SMS/email spam dataset:

- Load and inspect the data
- Class balance
- Message length and word-count distributions
- Most common words per class
- Sample messages

> **Prerequisite:** place a labeled dataset at `data/raw/spam.csv` with a `label` column (`ham`/`spam`) and a `message` column (the UCI v1/v2 layout is also accepted).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Allow importing the project's src modules regardless of where Jupyter was launched
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import RAW_DATA_DIR, load_raw_data  # noqa: E402

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
data_path = RAW_DATA_DIR / "sms_spam.csv"
if not data_path.exists():
    raise FileNotFoundError(
        f"No dataset found at {data_path}. "
        "Download the UCI SMS Spam Collection "
        "and save it as data/raw/sms_spam.csv with columns: label, message."
    )

df = load_raw_data()
df.head()

## 1. Basic overview

In [ ]:
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print("\n--- dtypes ---")
print(df.dtypes)
print("\n--- missing values ---")
print(df.isna().sum())
print(f"\nDuplicate messages: {df['message'].duplicated().sum():,}")

## 2. Class balance

In [ ]:
counts = df["label"].value_counts().sort_index()
counts.index = ["ham", "spam"]
counts_pct = df["label"].value_counts(normalize=True).sort_index() * 100
counts_pct.index = ["ham", "spam"]

pd.concat([counts, counts_pct.round(2)], axis=1, keys=["count", "percent"])

In [ ]:
ax = counts.plot(kind="bar", color=["#4caf50", "#f44336"], edgecolor="black")
for i, v in enumerate(counts):
    ax.text(i, v + v * 0.02, f"{v:,} ({counts_pct.iloc[i]:.1f}%)", ha="center")
ax.set_title("Class balance")
ax.set_xlabel("Label")
ax.set_ylabel("Number of messages")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 3. Message length

In [ ]:
df["char_len"] = df["message"].astype(str).str.len()
df["word_count"] = df["message"].astype(str).str.split().str.len()
df.groupby("label")[["char_len", "word_count"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ["char_len", "word_count"], ["Characters", "Words"]):
    for label, color in [(0, "#4caf50"), (1, "#f44336")]:
        subset = df.loc[df["label"] == label, col]
        ax.hist(subset, bins=50, alpha=0.5, color=color, label="ham" if label == 0 else "spam")
    ax.set_xlabel(title)
    ax.set_ylabel("Frequency")
    ax.set_xlim(0, 300)
    ax.legend()
    ax.set_title(f"Distribution of {title.lower()} by class")
plt.tight_layout()
plt.show()

## 4. Most common words

In [ ]:
from collections import Counter

def top_words(df, labels, n=20):
    counter = Counter()
    for msg in df.loc[df["label"].isin(labels), "message"].astype(str):
        counter.update(w for w in msg.lower().split() if w.isalpha())
    return counter.most_common(n)

ham_words = top_words(df, [0])
spam_words = top_words(df, [1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, words, title in zip(axes, [ham_words, spam_words], ["Ham", "Spam"]):
    labels, values = zip(*words)
    ax.barh(labels[::-1], values[::-1], color="#4caf50" if title == "Ham" else "#f44336")
    ax.set_title(f"Top words in {title} messages")
    ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

## 5. Sample messages

In [ ]:
print("--- HAM examples ---")
for msg in df.loc[df["label"] == 0, "message"].head(3):
    print(f"  - {msg}")

print("\n--- SPAM examples ---")
for msg in df.loc[df["label"] == 1, "message"].head(3):
    print(f"  - {msg}")

## Takeaways

Based on the exploration:

- Note the class balance — spam is typically a small minority, so accuracy alone is misleading; evaluate with precision, recall, and F1.
- Spam messages tend to be longer and packed with promotional words (`free`, `win`, `call`, `claim`, `text`).
- Duplicate messages are common in SMS spam datasets — decide whether to keep or drop them before training.
- Patterns worth engineering features for: URLs, numbers, all-caps words, exclamation marks, and money amounts.